In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import os
pd.options.mode.chained_assignment = None
import optuna
import time
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold, train_test_split
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, RobustScaler
import pickle
from collections import defaultdict


In [5]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Invierno'
    elif month in [3, 4, 5]:
        return 'Primavera'
    elif month in [6, 7, 8]:
        return 'Verano'
    else:
        return 'Otoño'
    

def get_zone(buoy):
    if buoy in ["CTD1", "CTD2", "CTD3", "CTD4"]:
        return 'Zona-1'
    elif buoy in ["CTD6", "CTD8", "CTD9", "CTD10", "CTD12"]:
        return 'Zona-2'
    elif buoy in ["CTD7"]:
        return 'Zona-3'
    elif buoy in ["CTD11"]:
        return 'Zona-4'

In [4]:
model_params ={
    "XGB" : {
        'n_estimators': 1000,
        'learning_rate': 0.02,
        'max_depth': 7,
        'min_child_weight': 2,
        'subsample': 0.7,
        'colsample_bytree': 0.7,
        'device': 'cpu',
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'enable_categorical': True,
        #'early_stopping_rounds': 50,
        'eval_metric': 'rmse'
        },

    "LBM" : {
        'learning_rate': 0.04,
        'num_leaves': 20,
        'max_depth': 7,
        'min_child_samples': 4,
        'subsample': 0.7,
        'colsample_bytree': 0.7,
        'n_estimators': 1000,
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'device': 'cpu',  
        'verbosity': -1,
        #'early_stopping_rounds': 50
        },

    "MLP": {
        'hidden_layer_sizes': (100,),
        'activation': 'relu',
        'solver': 'adam',
        'alpha': 0.0001,
        'learning_rate': 'constant',
        'learning_rate_init': 0.001,
        'max_iter': 200,
        'shuffle': True,
        'random_state': None,
        'tol': 1e-4,
        'n_iter_no_change': 25,
        'verbose': False,
        'early_stopping': True,
        'validation_fraction': 0.2
        },

    "SVR": {
        'kernel': 'rbf',        
        'C': 9.5,               
        'epsilon': 0.1,           
        'gamma': 'scale',        
        'shrinking': True,
        'tol': 1e-3,
        'max_iter': -1,          
        'verbose': False,
    },

    "KNN": {
        'n_neighbors': 5,
        'weights': 'distance',      
        'algorithm': 'auto',      
        'leaf_size': 25,
        'p': 2,                    
        'metric': 'minkowski',
        'n_jobs': -1             
    },

    "RF": {
        'n_estimators': 100,         
        'criterion': 'squared_error',
        'max_depth': 10,         
        'min_samples_split': 2,
        'min_samples_leaf': 2,    
        'bootstrap': True,
        'random_state': 42,
        'verbose': 0
    },

    "CAT": {
        'iterations': 1000,
        'learning_rate': 0.03,
        'depth': 6,
        'l2_leaf_reg': 3.0,
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'random_seed': 42,
        'allow_writing_files': False,
        'early_stopping_rounds': 50,
        'verbose': False
    },

    "ELN": {
        'alpha': 0.2,             
        'l1_ratio': 0.5,           
        'fit_intercept': True,
        'max_iter': 1000,
        'tol': 1e-4,
        'selection': 'cyclic',
        'random_state': 42
    }

}


models = {
    "XGB": XGBRegressor(**model_params['XGB']),
    "LBM": LGBMRegressor(**model_params['LBM']),
    "MLP": MLPRegressor(**model_params['MLP']),
    "SVR": SVR(**model_params['SVR']),
    "KNN": KNeighborsRegressor(**model_params['KNN']),
    "LR": LinearRegression(),
    "RF": RandomForestRegressor(**model_params['RF']),
    "CAT": CatBoostRegressor(**model_params["CAT"]),
    "ELN":  ElasticNet(**model_params["ELN"])
}

In [3]:
def cross_validation_training(dfs, depth):

    FOLDS = 5

    results = {}

    for nombre_df, df in list(dfs.items()):
    #for nombre_df, df in islice(dfs.items(), 3):
        print(f"\n=== Procesando {nombre_df} ===")
        # Ignoramos las columnas de Date, Lat, Lon y Buoy
        df = df.iloc[:, 4:]

        # Separamos el conjunto de datos en train y test: Train 75% Test 25%
        train, test = train_test_split(df, test_size=0.25, random_state=42, stratify=df["High_Chl"])
        # Seleccionamos la columna que queremos predecir
        target = "Chl"

        # Quitamos esa columna y el indicador de clorofila alta
        X = train.drop(columns=[target, "High_Chl"])
        # Para y cogemos solamente Chl
        y = train[target]
        # Para poder hacer StratifiedKFold y tener el mismo número de valores de Chl alta en cada fold
        y_class = train["High_Chl"]

        # Definimos X e y para test
        X_test = test.drop(columns=[target, "High_Chl"])
        y_test = test[target]

        # Dicts para guardar las predicciones sobre los conjuntos de validación, las y's correspondientes y los índices que corresponden dentro del loop de folds para el ensemble
        val_preds = {name: np.zeros(len(train)) for name in models}
        y_vals = defaultdict(list)
        val_indices = {}
        # Dict para guardar las predicciones sobre test
        test_preds = {name: np.zeros(len(test)) for name in models}

        # Stratified KFold de 5 folds
        skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

        # Dict para guardar resultados
        results[nombre_df] = {name: {'RMSE': [], 'R2': []} for name in models}

        # Loop para entrenar cada uno de los modelos
        for name, model in models.items():
            print(f"\n=== Training {name} ===")
            # Loop de folds, manteniendo la proporción de clases (Chl > 5) con y_class
            for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
                print(f"Fold {fold+1}")
                X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
                y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

                # Para modelos basados en distancias escalamos los datos
                if name in ["MLP", "SVR", "KNN", "LR", "ELN"]:
                    # Escalado dentro del loop de folds para evitar data leakage entre folds
                    scaler_X = RobustScaler()
                    scaler_y = RobustScaler()
                    X_train_scaled = scaler_X.fit_transform(X_train)
                    X_val_scaled = scaler_X.transform(X_val)
                    X_test_scaled = scaler_X.transform(X_test)
                    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
                    # Entrenamos modelo con datos escalados
                    model.fit(X_train_scaled, y_train_scaled)
                    # Predicción sobre val y test, haciendo la transformada inversa para devolver y a su escala
                    val_pred = scaler_y.inverse_transform(model.predict(X_val_scaled).reshape(-1, 1)).ravel()
                    test_pred = scaler_y.inverse_transform(model.predict(X_test_scaled).reshape(-1, 1)).ravel()
                # Para modelos basados en árboles no es necesario escalar
                else:
                    # Entrenamos el modelo
                    model.fit(X_train, y_train)
                    # Predicción sobre val y test
                    val_pred = model.predict(X_val)
                    test_pred = model.predict(X_test)

                # Guardamos las predicciones sobre val, las y's que les corresponden y los índices
                val_preds[name][val_idx] = val_pred
                if name == list(models.keys())[0]:
                    # Solo lo guardamos una vez
                    y_vals[fold] = y_val
                    val_indices[fold] = val_idx  # val_idx es un array de índices relativos a train

                # Guardamos la predicción de test, haciendo la media entre los folds
                test_preds[name] += test_pred / FOLDS

                # Calculamos y guardamos métricas
                rmse = np.sqrt(mean_squared_error(y_val, val_pred))
                r2 = r2_score(y_val, val_pred)
                results[nombre_df][name]['RMSE'].append(rmse)
                results[nombre_df][name]['R2'].append(r2)

        # Extendemos el dict de resultados con el ensemble
        results[nombre_df]["ENS"] = {'RMSE': [], 'R2': []}

        for fold in range(FOLDS):
            # Índices y valores del fold actual
            fold_val_idx = val_indices[fold]
            meta_X_val = np.vstack([val_preds[model][fold_val_idx] for model in models]).T
            meta_y_val = y_vals[fold]

            # Índices de entrenamiento: todos menos el fold actual
            train_folds = [i for i in range(FOLDS) if i != fold]
            train_idx = np.concatenate([val_indices[i] for i in train_folds])
            meta_X_train = np.vstack([val_preds[model][train_idx] for model in models]).T
            meta_y_train = y.iloc[train_idx]

            # Entrenamos el meta-modelo solo con los otros 4 folds
            meta_model = Ridge().fit(meta_X_train, meta_y_train)

            # Predicción en el fold actual (no visto)
            ensemble_pred = meta_model.predict(meta_X_val)

            rmse = np.sqrt(mean_squared_error(meta_y_val, ensemble_pred))
            r2 = r2_score(meta_y_val, ensemble_pred)
            results[nombre_df]["ENS"]['RMSE'].append(rmse)
            results[nombre_df]["ENS"]['R2'].append(r2)



    # === Evaluación final sobre test ===
        for name in models:
            rmse_test = np.sqrt(mean_squared_error(y_test, test_preds[name]))
            r2_test = r2_score(y_test, test_preds[name])
            results[nombre_df][name]["RMSE test"] = rmse_test
            results[nombre_df][name]["R2 test"] = r2_test

        # Construcción del meta-modelo sobre todo el conjunto de validación
        final_meta_X = np.vstack([val_preds[model] for model in models]).T
        final_meta_y = y.values
        ensemble_model = Ridge().fit(final_meta_X, final_meta_y)

        # Predicción sobre test del ensemble
        meta_X_test = np.vstack([test_preds[model] for model in models]).T
        ensemble_test_pred = ensemble_model.predict(meta_X_test)
        # Evaluación del ensemble sobre test
        rmse_ens_test = np.sqrt(mean_squared_error(y_test, ensemble_test_pred))
        r2_ens_test = r2_score(y_test, ensemble_test_pred)
        results[nombre_df]["ENS"]["RMSE test"] = rmse_ens_test
        results[nombre_df]["ENS"]["R2 test"] = r2_ens_test

    with open(f"training_results/results_entrenamiento_CV_{depth}.pkl", "wb") as f:
        pickle.dump(results, f)

    return results

In [6]:
depths = ["eq_0", "eq_1", "lt_1", "lt_2", "gt_1", "gt_3"]
for depth in depths: 
    # Cargamos los csv de los tifs
    path = "saved_files/dataset"
    dfs = {}
    all_datasets = False
    for archivo in os.listdir(path):
        if all_datasets:
            if f"{depth}" in archivo:
                nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
                ruta_completa = os.path.join(path, archivo)
                dfs[nombre_sin_extension] = pd.read_csv(ruta_completa)
        else:
            if archivo.endswith(f"{depth}_features.csv"):
                nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
                ruta_completa = os.path.join(path, archivo)
                dfs[nombre_sin_extension[:-9]] = pd.read_csv(ruta_completa)

    # Limpiamos valores nulos
    for nombre_df, df in dfs.items():
        for band_set in ["rhow", "rhown","rtoa"]:
            dfs[nombre_df] = df.dropna()

    
    for nombre_df, df in dfs.items():
        # Marcamos las columnas de CHl alta (equivalente a quantile(0.93))
        df["High_Chl"] = df["Chl"]>5
        # Sacamos la estación de cada fecha
        df['Date'] = pd.to_datetime(df['Date'])
        df['Season'] = df['Date'].dt.month.apply(get_season)
        
        # Etiquetamos la zona de la observación (comentado porque para aplicar el modelo habría que segmentar todo el Mar Menor - se puede hacer por px)
        # df['Zone'] = df['Buoy'].apply(get_zone)
        # Ponemos las columnas como categóricas, para Season y Zone
        for col in df.select_dtypes(include='object').columns:
            df[col] = df[col].astype('category')

        df = pd.concat([df.drop(columns=["Season"]),pd.get_dummies(df["Season"])], axis=1)
        dfs[nombre_df] = df

    results = cross_validation_training(dfs, depth)


=== Procesando C2X-Complex_rhow_9x9_depth_eq_0 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 2
Fold 3
Fold 4
Fold 5

=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando C2RCC_rhow_1x1_depth_eq_0 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 2
Fold 3


...


=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando TOA_3x3_depth_gt_3 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando C2X-Complex_rhow_1x1_depth_gt_3 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5



/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando C2X_rhow_1x1_depth_gt_3 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 3
Fold 4
Fold 5

=== Training SVR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LR ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training ELN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


In [122]:
FOLDS = 5

results = {}

for nombre_df, df in list(dfs.items()):
#for nombre_df, df in islice(dfs.items(), 3):
    print(f"\n=== Procesando {nombre_df} ===")
    # Ignoramos las columnas de Date, Lat, Lon y Buoy
    df = df.iloc[:, 4:]

    # Separamos el conjunto de datos en train y test: Train 75% Test 25%
    train, test = train_test_split(df, test_size=0.25, random_state=42, stratify=df["High_Chl"])
    # Seleccionamos la columna que queremos predecir
    target = "Chl"

    # Quitamos esa columna y el indicador de clorofila alta
    X = train.drop(columns=[target, "High_Chl"])
    # Para y cogemos solamente Chl
    y = train[target]
    # Para poder hacer StratifiedKFold y tener el mismo número de valores de Chl alta en cada fold
    y_class = train["High_Chl"]

    # Definimos X e y para test
    X_test = test.drop(columns=[target, "High_Chl"])
    y_test = test[target]

    # Dicts para guardar las predicciones sobre los conjuntos de validación, las y's correspondientes y los índices que corresponden dentro del loop de folds para el ensemble
    val_preds = {name: np.zeros(len(train)) for name in models}
    y_vals = defaultdict(list)
    val_indices = {}
    # Dict para guardar las predicciones sobre test
    test_preds = {name: np.zeros(len(test)) for name in models}

    # Stratified KFold de 5 folds
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

    # Dict para guardar resultados
    results[nombre_df] = {name: {'RMSE': [], 'R2': []} for name in models}

    # Loop para entrenar cada uno de los modelos
    for name, model in models.items():
        print(f"\n=== Training {name} ===")
        # Loop de folds, manteniendo la proporción de clases (Chl > 5) con y_class
        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
            print(f"Fold {fold+1}")
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            # Para modelos basados en distancias escalamos los datos
            if name in ["MLP", "SVR", "KNN", "LR", "ELN"]:
                # Escalado dentro del loop de folds para evitar data leakage entre folds
                scaler_X = RobustScaler()
                scaler_y = RobustScaler()
                X_train_scaled = scaler_X.fit_transform(X_train)
                X_val_scaled = scaler_X.transform(X_val)
                X_test_scaled = scaler_X.transform(X_test)
                y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
                # Entrenamos modelo con datos escalados
                model.fit(X_train_scaled, y_train_scaled)
                # Predicción sobre val y test, haciendo la transformada inversa para devolver y a su escala
                val_pred = scaler_y.inverse_transform(model.predict(X_val_scaled).reshape(-1, 1)).ravel()
                test_pred = scaler_y.inverse_transform(model.predict(X_test_scaled).reshape(-1, 1)).ravel()
            # Para modelos basados en árboles no es necesario escalar
            else:
                # Entrenamos el modelo
                model.fit(X_train, y_train)
                # Predicción sobre val y test
                val_pred = model.predict(X_val)
                test_pred = model.predict(X_test)

            # Guardamos las predicciones sobre val, las y's que les corresponden y los índices
            val_preds[name][val_idx] = val_pred
            if name == list(models.keys())[0]:
                # Solo lo guardamos una vez
                y_vals[fold] = y_val
                val_indices[fold] = val_idx  # val_idx es un array de índices relativos a train

            # Guardamos la predicción de test, haciendo la media entre los folds
            test_preds[name] += test_pred / FOLDS

            # Calculamos y guardamos métricas
            rmse = np.sqrt(mean_squared_error(y_val, val_pred))
            r2 = r2_score(y_val, val_pred)
            results[nombre_df][name]['RMSE'].append(rmse)
            results[nombre_df][name]['R2'].append(r2)

    # Extendemos el dict de resultados con el ensemble
    results[nombre_df]["ENS"] = {'RMSE': [], 'R2': []}

    for fold in range(FOLDS):
        # Índices y valores del fold actual
        fold_val_idx = val_indices[fold]
        meta_X_val = np.vstack([val_preds[model][fold_val_idx] for model in models]).T
        meta_y_val = y_vals[fold]

        # Índices de entrenamiento: todos menos el fold actual
        train_folds = [i for i in range(FOLDS) if i != fold]
        train_idx = np.concatenate([val_indices[i] for i in train_folds])
        meta_X_train = np.vstack([val_preds[model][train_idx] for model in models]).T
        meta_y_train = y.iloc[train_idx]

        # Entrenamos el meta-modelo solo con los otros 4 folds
        meta_model = Ridge().fit(meta_X_train, meta_y_train)

        # Predicción en el fold actual (no visto)
        ensemble_pred = meta_model.predict(meta_X_val)

        rmse = np.sqrt(mean_squared_error(meta_y_val, ensemble_pred))
        r2 = r2_score(meta_y_val, ensemble_pred)
        results[nombre_df]["ENS"]['RMSE'].append(rmse)
        results[nombre_df]["ENS"]['R2'].append(r2)



# === Evaluación final sobre test ===
    for name in models:
        rmse_test = np.sqrt(mean_squared_error(y_test, test_preds[name]))
        r2_test = r2_score(y_test, test_preds[name])
        results[nombre_df][name]["RMSE test"] = rmse_test
        results[nombre_df][name]["R2 test"] = r2_test

    # Construcción del meta-modelo sobre todo el conjunto de validación
    final_meta_X = np.vstack([val_preds[model] for model in models]).T
    final_meta_y = y.values
    ensemble_model = Ridge().fit(final_meta_X, final_meta_y)

    # Predicción sobre test del ensemble
    meta_X_test = np.vstack([test_preds[model] for model in models]).T
    ensemble_test_pred = ensemble_model.predict(meta_X_test)
    # Evaluación del ensemble sobre test
    rmse_ens_test = np.sqrt(mean_squared_error(y_test, ensemble_test_pred))
    r2_ens_test = r2_score(y_test, ensemble_test_pred)
    results[nombre_df]["ENS"]["RMSE test"] = rmse_ens_test
    results[nombre_df]["ENS"]["R2 test"] = r2_ens_test



=== Procesando C2X-Complex_rhown_5x5_depth_lt_2 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training EN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando TOA_9x9_depth_lt_2 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2
Fold 3
Fold 4


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training EN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando C2X_rhown_5x5_depth_lt_2 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training EN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando C2RCC_rhow_1x1_depth_lt_2 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1
Fold 2


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 3
Fold 4
Fold 5


...


=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training EN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Procesando TOA_5x5_depth_lt_2 ===

=== Training XGB ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training LBM ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training MLP ===
Fold 1


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 2
Fold 3
Fold 4


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 5

=== Training KNN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training RF ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training CAT ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

=== Training EN ===
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


In [38]:
with open(f"training_results/results_entrenamiento_CV_{depth}.pkl", "rb") as f:
    results = pickle.load(f)

In [128]:
rows = []

for df_name, model_scores in results.items():
    row = {}
    for model_name, metrics in model_scores.items():
        for metric_name, values in metrics.items():
            if isinstance(values, list):  # Solo para los que tienen listas (folds)
                mean_val = np.mean(values)
                std_val = np.std(values)
                row[(metric_name, model_name)] = f"{mean_val:.2f} ± {std_val:.2f}"
            else:
                # Para el ensemble que tiene un único valor
                row[(metric_name, model_name)] = f"{values:.2f}"
    rows.append((df_name, row))

df_results = pd.DataFrame.from_dict(dict(rows), orient="index")
df_results.columns = pd.MultiIndex.from_tuples(df_results.columns, names=["Metric", "Model"])
df_results = df_results.sort_index(axis=1, level=0)
df_results = df_results.sort_index(axis=0)

In [129]:
df_results

Metric                                     R2                            \
Model                                     CAT           EN     Ensemble   
C2RCC_rhow_1x1_depth_lt_2         0.69 ± 0.08  0.42 ± 0.05  0.75 ± 0.08   
C2RCC_rhow_3x3_depth_lt_2         0.68 ± 0.05  0.41 ± 0.03  0.66 ± 0.11   
C2RCC_rhow_5x5_depth_lt_2         0.63 ± 0.12  0.46 ± 0.04  0.57 ± 0.11   
C2RCC_rhow_9x9_depth_lt_2         0.57 ± 0.20  0.42 ± 0.03  0.55 ± 0.15   
C2RCC_rhown_1x1_depth_lt_2        0.72 ± 0.09  0.41 ± 0.05  0.74 ± 0.08   
C2RCC_rhown_3x3_depth_lt_2        0.69 ± 0.05  0.40 ± 0.03  0.70 ± 0.12   
C2RCC_rhown_5x5_depth_lt_2        0.66 ± 0.09  0.45 ± 0.04  0.58 ± 0.14   
C2RCC_rhown_9x9_depth_lt_2        0.59 ± 0.19  0.42 ± 0.03  0.61 ± 0.16   
C2X-Complex_rhow_1x1_depth_lt_2   0.60 ± 0.10  0.43 ± 0.03  0.60 ± 0.05   
C2X-Complex_rhow_3x3_depth_lt_2   0.71 ± 0.05  0.35 ± 0.12  0.70 ± 0.07   
C2X-Complex_rhow_5x5_depth_lt_2   0.67 ± 0.06  0.38 ± 0.11  0.61 ± 0.07   
C2X-Complex_rhow_9x9_depth_lt_2   0.74 ± 0.14  0.42 ± 0.04  0.74 ± 0.11   
C2X-Complex_rhown_1x1_depth_lt_2  0.61 ± 0.13  0.44 ± 0.04  0.58 ± 0.12   
C2X-Complex_rhown_3x3_depth_lt_2  0.69 ± 0.04  0.34 ± 0.12  0.64 ± 0.07   
C2X-Complex_rhown_5x5_depth_lt_2  0.67 ± 0.06  0.39 ± 0.09  0.60 ± 0.10   
C2X-Complex_rhown_9x9_depth_lt_2  0.66 ± 0.12  0.42 ± 0.03  0.64 ± 0.09   
C2X_rhow_1x1_depth_lt_2           0.59 ± 0.06  0.50 ± 0.05  0.49 ± 0.16   
C2X_rhow_3x3_depth_lt_2           0.52 ± 0.05  0.36 ± 0.15  0.43 ± 0.20   
C2X_rhow_5x5_depth_lt_2           0.63 ± 0.11  0.40 ± 0.12  0.65 ± 0.10   
C2X_rhow_9x9_depth_lt_2           0.60 ± 0.10  0.47 ± 0.04  0.54 ± 0.15   
C2X_rhown_1x1_depth_lt_2          0.50 ± 0.14  0.47 ± 0.03  0.42 ± 0.15   
C2X_rhown_3x3_depth_lt_2          0.45 ± 0.10  0.27 ± 0.25  0.39 ± 0.13   
C2X_rhown_5x5_depth_lt_2          0.53 ± 0.10  0.33 ± 0.17  0.53 ± 0.13   
C2X_rhown_9x9_depth_lt_2          0.57 ± 0.10  0.46 ± 0.04  0.51 ± 0.23   
TOA_1x1_depth_lt_2                0.67 ± 0.19  0.08 ± 0.11  0.69 ± 0.15   
TOA_3x3_depth_lt_2                0.70 ± 0.19  0.08 ± 0.11  0.63 ± 0.23   
TOA_5x5_depth_lt_2                0.68 ± 0.20  0.09 ± 0.12  0.65 ± 0.21   
TOA_9x9_depth_lt_2                0.68 ± 0.21  0.08 ± 0.12  0.71 ± 0.16   

Metric                                                                   \
Model                                     KNN          LBM          MLP   
C2RCC_rhow_1x1_depth_lt_2         0.78 ± 0.06  0.62 ± 0.15  0.64 ± 0.06   
C2RCC_rhow_3x3_depth_lt_2         0.66 ± 0.13  0.48 ± 0.25  0.61 ± 0.12   
C2RCC_rhow_5x5_depth_lt_2         0.67 ± 0.07  0.52 ± 0.20  0.63 ± 0.14   
C2RCC_rhow_9x9_depth_lt_2         0.57 ± 0.18  0.43 ± 0.26  0.55 ± 0.15   
C2RCC_rhown_1x1_depth_lt_2        0.78 ± 0.05  0.65 ± 0.10  0.68 ± 0.08   
C2RCC_rhown_3x3_depth_lt_2        0.67 ± 0.12  0.50 ± 0.19  0.48 ± 0.13   
C2RCC_rhown_5x5_depth_lt_2        0.67 ± 0.10  0.53 ± 0.19  0.65 ± 0.08   
C2RCC_rhown_9x9_depth_lt_2        0.55 ± 0.14  0.50 ± 0.28  0.56 ± 0.18   
C2X-Complex_rhow_1x1_depth_lt_2   0.62 ± 0.07  0.46 ± 0.17  0.52 ± 0.06   
C2X-Complex_rhow_3x3_depth_lt_2   0.62 ± 0.08  0.63 ± 0.08  0.56 ± 0.14   
C2X-Complex_rhow_5x5_depth_lt_2   0.61 ± 0.12  0.58 ± 0.10  0.44 ± 0.11   
C2X-Complex_rhow_9x9_depth_lt_2   0.65 ± 0.07  0.58 ± 0.34  0.59 ± 0.08   
C2X-Complex_rhown_1x1_depth_lt_2  0.64 ± 0.09  0.48 ± 0.15  0.57 ± 0.06   
C2X-Complex_rhown_3x3_depth_lt_2  0.58 ± 0.06  0.65 ± 0.09  0.58 ± 0.07   
C2X-Complex_rhown_5x5_depth_lt_2  0.58 ± 0.13  0.61 ± 0.13  0.53 ± 0.09   
C2X-Complex_rhown_9x9_depth_lt_2  0.61 ± 0.08  0.55 ± 0.21  0.54 ± 0.20   
C2X_rhow_1x1_depth_lt_2           0.54 ± 0.07  0.56 ± 0.17  0.54 ± 0.08   
C2X_rhow_3x3_depth_lt_2           0.47 ± 0.19  0.49 ± 0.15  0.20 ± 0.35   
C2X_rhow_5x5_depth_lt_2           0.55 ± 0.20  0.63 ± 0.09  0.47 ± 0.13   
C2X_rhow_9x9_depth_lt_2           0.62 ± 0.13  0.47 ± 0.14  0.54 ± 0.07   
C2X_rhown_1x1_depth_lt_2          0.49 ± 0.08  0.53 ± 0.15  0.47 ± 0.16   
C2X_rhown_3x3_depth_lt_2

In [130]:
# Asumiendo que tu DataFrame se llama df_results
df_sorted = df_results["R2 test"].copy()

# Añadir una columna auxiliar con el R2 máximo por fila
df_sorted["max_R2"] = df_sorted.max(axis=1)

# Ordenar por esa columna en orden descendente
df_sorted = df_sorted.sort_values("max_R2", ascending=False)

# Eliminar la columna auxiliar
df_sorted = df_sorted.drop(columns="max_R2")



In [131]:
df_sorted

Model,CAT,EN,Ensemble,KNN,LBM,MLP,RF,XGB
C2RCC_rhow_9x9_depth_lt_2,0.85,0.37,0.79,0.78,0.78,0.60,0.70,0.81
C2RCC_rhown_9x9_depth_lt_2,0.85,0.36,0.80,0.76,0.77,0.59,0.73,0.82
C2X-Complex_rhow_5x5_depth_lt_2,0.84,0.43,0.82,0.70,0.68,0.58,0.69,0.76
C2RCC_rhow_5x5_depth_lt_2,0.82,0.36,0.78,0.80,0.75,0.68,0.66,0.78
C2RCC_rhown_5x5_depth_lt_2,0.81,0.35,0.81,0.77,0.75,0.65,0.65,0.77
C2X-Complex_rhown_5x5_depth_lt_2,0.80,0.44,0.70,0.75,0.57,0.60,0.59,0.61
C2X_rhow_5x5_depth_lt_2,0.78,0.53,0.80,0.69,0.74,0.60,0.73,0.72
C2RCC_rhown_3x3_depth_lt_2,0.76,0.32,0.80,0.74,0.71,0.55,0.60,0.71
C2X-Complex_rhow_9x9_depth_lt_2,0.79,0.45,0.74,0.65,0.69,0.67,0.66,0.71
TOA_9x9_depth_lt_2,0.79,-0.01,0.71,0.75,0.71,0.52,0.69,0.74


In [84]:
df_sorted

Model,CAT,EN,Ensemble,KNN,LBM,MLP,RF,XGB
C2RCC_rhow_5x5_depth_lt_1,0.76,0.37,0.77,0.73,0.73,0.68,0.72,0.76
TOA_9x9_depth_lt_1,0.70,0.09,0.70,0.60,0.54,0.57,0.58,0.56
C2X-Complex_rhown_3x3_depth_lt_1,0.69,0.38,0.65,0.62,0.63,0.45,0.61,0.65


In [40]:
df_results["R2 test"]

Model,CAT,EN,Ensemble,KNN,LBM,LR,MLP,RF,SVR,XGB
C2RCC_rhow_1x1_depth_lt_1,0.74,0.46,0.74,0.72,0.70,0.62,0.61,0.69,0.69,0.75
C2RCC_rhow_3x3_depth_lt_1,0.71,0.42,0.66,0.66,0.72,0.39,0.56,0.67,0.72,0.71
C2RCC_rhow_5x5_depth_lt_1,0.76,0.49,0.76,0.73,0.76,0.39,0.65,0.71,0.77,0.78
C2RCC_rhow_9x9_depth_lt_1,0.81,0.48,0.76,0.70,0.77,0.63,0.69,0.72,0.75,0.80
C2RCC_rhown_1x1_depth_lt_1,0.73,0.45,0.77,0.76,0.69,0.53,0.62,0.68,0.68,0.75
C2RCC_rhown_3x3_depth_lt_1,0.74,0.42,0.64,0.63,0.71,0.43,0.67,0.65,0.71,0.72
C2RCC_rhown_5x5_depth_lt_1,0.76,0.48,0.71,0.69,0.76,0.56,0.67,0.72,0.75,0.79
C2RCC_rhown_9x9_depth_lt_1,0.79,0.48,0.72,0.68,0.77,0.53,0.63,0.73,0.75,0.77
C2X-Complex_rhow_1x1_depth_lt_1,0.73,0.53,0.78,0.75,0.68,0.43,0.62,0.64,0.62,0.75
C2X-Complex_rhow_3x3_depth_lt_1,0.72,0.39,0.74,0.68,0.64,0.11,0.61,0.64,0.73,0.66


In [29]:
df_results["R2"]

Model,CAT,EN,Ensemble,KNN,LBM,LR,MLP,RF,SVR,XGB
C2RCC_rhow_1x1_depth_gt_3,0.39 ± 0.10,0.25 ± 0.06,0.55 ± 0.09,0.35 ± 0.12,0.38 ± 0.11,-0.54 ± 1.52,0.39 ± 0.08,0.35 ± 0.10,0.37 ± 0.11,0.37 ± 0.08
C2RCC_rhow_3x3_depth_gt_3,0.40 ± 0.13,0.27 ± 0.05,0.61 ± 0.07,0.39 ± 0.10,0.40 ± 0.10,0.24 ± 0.18,0.35 ± 0.08,0.42 ± 0.13,0.40 ± 0.10,0.38 ± 0.15
C2RCC_rhow_5x5_depth_gt_3,0.48 ± 0.13,0.29 ± 0.05,0.65 ± 0.10,0.46 ± 0.11,0.45 ± 0.13,0.08 ± 0.58,0.40 ± 0.05,0.46 ± 0.13,0.48 ± 0.09,0.47 ± 0.14
C2RCC_rhow_9x9_depth_gt_3,0.51 ± 0.08,0.29 ± 0.05,0.68 ± 0.09,0.42 ± 0.12,0.50 ± 0.09,0.30 ± 0.17,0.39 ± 0.05,0.48 ± 0.08,0.54 ± 0.06,0.49 ± 0.09
C2RCC_rhown_1x1_depth_gt_3,0.39 ± 0.09,0.25 ± 0.06,0.58 ± 0.08,0.35 ± 0.10,0.34 ± 0.11,0.23 ± 0.08,0.32 ± 0.08,0.38 ± 0.11,0.38 ± 0.11,0.42 ± 0.08
C2RCC_rhown_3x3_depth_gt_3,0.40 ± 0.11,0.27 ± 0.05,0.59 ± 0.12,0.40 ± 0.09,0.44 ± 0.13,0.22 ± 0.14,0.42 ± 0.09,0.43 ± 0.10,0.42 ± 0.10,0.39 ± 0.14
C2RCC_rhown_5x5_depth_gt_3,0.46 ± 0.11,0.29 ± 0.05,0.70 ± 0.06,0.50 ± 0.11,0.53 ± 0.09,0.20 ± 0.22,0.47 ± 0.05,0.49 ± 0.10,0.49 ± 0.09,0.50 ± 0.09
C2RCC_rhown_9x9_depth_gt_3,0.52 ± 0.10,0.29 ± 0.05,0.70 ± 0.06,0.45 ± 0.13,0.48 ± 0.12,0.05 ± 0.47,0.38 ± 0.07,0.51 ± 0.09,0.56 ± 0.06,0.50 ± 0.11
C2X-Complex_rhow_1x1_depth_gt_3,0.45 ± 0.08,0.19 ± 0.19,0.66 ± 0.04,0.34 ± 0.07,0.33 ± 0.13,-1.95 ± 3.97,0.36 ± 0.07,0.39 ± 0.10,0.41 ± 0.07,0.45 ± 0.08
C2X-Complex_rhow_3x3_depth_gt_3,0.48 ± 0.05,0.28 ± 0.02,0.71 ± 0.04,0.45 ± 0.07,0.34 ± 0.07,0.29 ± 0.16,0.47 ± 0.05,0.44 ± 0.08,0.46 ± 0.12,0.39 ± 0.12
